# RL Gaming

---

## Setup

In [ ]:
from pathlib import Path

from custom_callback import CustomCallback
from env_optimiser import EnvOptimiser
from feature_extractor import MinigridFeaturesExtractor
from levels_utils import (
    bar_plot_levels_success_rate,
    display_all_levels,
    display_procedural_levels,
    evaluate_all_levels_with_model_ppo,
    evaluate_all_levels_with_model_recurrent_ppo,
    test_ppo_model_on_level,
)
from minigrid.wrappers import ImgObsWrapper, OneHotPartialObsWrapper
from minigrid_levels_env import MiniGridLevelsEnv
from plot_utils import plot_training_results
from procedural_level import ProceduralLevel
from sb3_contrib import RecurrentPPO
from stable_baselines3 import PPO

PPO_MODEL_PATH = Path("../../data/ppo/model.zip")
RECURRENT_PPO_MODEL_PATH = Path("../../data/rppo/model.zip")

---

## Environment

### Random training levels

In [ ]:
display_procedural_levels(n_levels=6, max_cols=3)

### Custom testing levels

In [ ]:
display_all_levels(max_cols=3)

---

## PPO

### Training

In [ ]:
n_envs = 8
n_timesteps = 200_000
freq = 100
eval_freq = max(freq // n_envs, 1)

save_dir = Path("../../models/")
save_dir.mkdir(parents=True, exist_ok=True)

env = MiniGridLevelsEnv(level_id=1)
optimiser = EnvOptimiser(env=env, n_envs=n_envs, wrapper_cls=[ImgObsWrapper], save_dir=save_dir)
vec_env_train = optimiser.build_vec_env()
policy_kwargs = {
    "features_extractor_class": MinigridFeaturesExtractor,
    "features_extractor_kwargs": {"features_dim": 128},
}
model_dir = optimiser.file_path.parent
callback = CustomCallback(check_freq=eval_freq, save_dir=model_dir, verbose=0)
model = PPO(policy="CnnPolicy", env=vec_env_train, policy_kwargs=policy_kwargs)
model.learn(total_timesteps=n_timesteps, callback=callback, progress_bar=True)

### Plots

In [4]:
plot_training_results(log_dir=PPO_MODEL_PATH.parent)

NameError: name 'PPO_MODEL_PATH' is not defined

## RecurrentPPO

### Training

In [ ]:
n_envs = 8
n_timesteps = 5_000
freq = 100
eval_freq = max(freq // n_envs, 1)

save_dir = Path("../../models/")
save_dir.mkdir(parents=True, exist_ok=True)

env = ProceduralLevel(difficulty=1000, max_steps=200)
optimiser = EnvOptimiser(
    env=env, n_envs=n_envs, wrapper_cls=[OneHotPartialObsWrapper, ImgObsWrapper], save_dir=save_dir
)
vec_env_train = optimiser.build_vec_env()
policy_kwargs = {
    "features_extractor_class": MinigridFeaturesExtractor,
    "features_extractor_kwargs": {"features_dim": 128},
}
model_dir = optimiser.file_path.parent
callback = CustomCallback(check_freq=eval_freq, save_dir=model_dir, verbose=0)
model = RecurrentPPO(policy="CnnLstmPolicy", env=vec_env_train, policy_kwargs=policy_kwargs)
model.learn(total_timesteps=n_timesteps, callback=callback, progress_bar=True)

### Plots

In [ ]:
plot_training_results(log_dir=RECURRENT_PPO_MODEL_PATH.parent)

---

## Testing

In [ ]:
model_actions_results = evaluate_all_levels_with_model_ppo(PPO_MODEL_PATH)
bar_plot_levels_success_rate(model_actions_results)

In [ ]:
test_ppo_model_on_level(model_path=PPO_MODEL_PATH, level_id=1)

In [ ]:
model_actions_results = evaluate_all_levels_with_model_recurrent_ppo(RECURRENT_PPO_MODEL_PATH)
bar_plot_levels_success_rate(model_actions_results)